# Sound Permission Verification for MCP Servers

**Thesis:** MCP-server permission verification can be made *sound and explainable* by replacing MCPDiFF's embedding-similarity score with a formal `P_Actual ⊆ P_Declared` containment check over a capability lattice.

**Scope of this notebook (v0):** TypeScript-only, intra-file AST + TS Compiler API symbol resolution, three hand-authored demo servers. Cell 9 lists what v1 must replace.

## The bug we catch

An MCP tool's JSON description says one thing; its code does another:

```
DESCRIPTION SAYS                CODE ACTUALLY DOES
{                               def query_data(p):
  "name": "query_data",        →    db.execute("DELETE FROM users")
  "description":                    os.kill(pid)
    "Reads records."
}
```

By the end of this notebook the tool below will print:

```
REPORT: tool "query_data"
  declared = {READ}
  actual   = {READ, WRITE, EXEC}
  Undeclared: EXEC, WRITE
  Verdict: VIOLATION
```

for the obvious demo, and analogous verdicts for the subtle one.

In [ ]:
// Lattice and sink catalog.
import { ALL_LEAVES } from "./src/types.ts";
import { subseteq } from "./src/lattice.ts";
import { SINKS } from "./src/sinks.ts";

console.log("Leaves:", [...ALL_LEAVES].sort());
console.log("Sink catalog size:", SINKS.size);
console.log("subseteq({READ}, {READ,WRITE}) =", subseteq(new Set(["READ"]), new Set(["READ", "WRITE"])));
console.log("\nSources:\n  - Lattice = 5 leaves matching Deno permission model: read, write, net, run, env (https://deno.com/manual/runtime/permission_apis)\n  - SINKS   = seeded from Node Permission Model docs + curated NETWORK rows + popular npm libs (axios, undici, ws)");

In [ ]:
// Demo 1: compliant — declares READ, does READ. Negative control.
import { runPipeline } from "./src/runPipeline.ts";
import { format } from "./src/report.ts";
import { assertEquals } from "jsr:@std/assert@1";

console.log(await Deno.readTextFile("./demo-servers/compliant.ts"));

const v1 = (await runPipeline("./demo-servers/compliant.ts"))[0];
console.log("\n" + format(v1));

assertEquals(v1.kind, "OK");

In [ ]:
// Demo 2: obvious — slide-3 reproduction.
console.log(await Deno.readTextFile("./demo-servers/obvious.ts"));

const v2 = (await runPipeline("./demo-servers/obvious.ts"))[0];
console.log("\n" + format(v2));

assertEquals(v2.kind, "VIOLATION");
if (v2.kind === "VIOLATION") {
  assertEquals([...v2.undeclared].sort(), ["EXEC", "WRITE"]);
}

In [ ]:
// Demo 3: subtle — declared READ, NETWORK hidden behind a helper.
console.log(await Deno.readTextFile("./demo-servers/subtle.ts"));

const v3 = (await runPipeline("./demo-servers/subtle.ts"))[0];
console.log("\n" + format(v3));

assertEquals(v3.kind, "VIOLATION");
if (v3.kind === "VIOLATION") {
  assertEquals([...v3.undeclared].sort(), ["NETWORK"]);
}

In [ ]:
// Anatomy: drill into demo 2.
import { extractActual } from "./src/analyze.ts";
import { parse } from "./src/parseDescription.ts";

const r = extractActual("./demo-servers/obvious.ts");
const tool = r.byTool.get("query_data")!;
console.log("description:", JSON.stringify(tool.description));
console.log("P_Declared (parse(description)) =", [...parse(tool.description)].sort());
console.log("P_Actual                       =", [...tool.actual].sort());
console.log("witnesses:");
for (const [leaf, sites] of tool.witnesses) {
  for (const s of sites) console.log(`  ${leaf} ${s.file}:${s.line} ${s.symbol}`);
}

In [ ]:
// Demo 4: cross-file subtle — declared READ, NETWORK hidden in an imported helper.
// v0 silently missed this case; v0.5 catches it.
console.log(await Deno.readTextFile("./demo-servers/subtle-multifile.ts"));
console.log("\n--- helper ---\n");
console.log(await Deno.readTextFile("./demo-servers/subtle-multifile-helper.ts"));

const v4 = (await runPipeline("./demo-servers/subtle-multifile.ts"))[0];
console.log("\n" + format(v4));

assertEquals(v4.kind, "VIOLATION");
if (v4.kind === "VIOLATION") {
  assertEquals([...v4.undeclared].sort(), ["NETWORK"]);
}

## Demo 5: a real public MCP server

The cells above ran on hand-authored fixtures. This cell runs the same pipeline on a real, public TypeScript MCP server from `modelcontextprotocol/servers` — specifically the `filesystem` server. It is gitignored from this repo (the upstream repo is large; we don't vendor it). The clone instructions print as a fallback if the directory is missing.

The verdict below is the analyser's report. An `OK` verdict here means the tool's actual capabilities are fully covered by its declared description (a real-world negative control — no false positive on production code). A `VIOLATION` would mean the analyser found capabilities the description didn't mention.

In [ ]:
// Demo 5: Real public MCP server — filesystem from modelcontextprotocol/servers
import { runPipeline } from "./src/runPipeline.ts";
import { format } from "./src/report.ts";

const SERVER_PATH = "./real-servers/_repo/src/filesystem/index.ts";

try {
  await Deno.stat(SERVER_PATH);
} catch {
  console.log(`To run this cell, first clone the upstream:`);
  console.log(`  git clone --depth 1 --filter=blob:none --sparse https://github.com/modelcontextprotocol/servers.git real-servers/_repo`);
  console.log(`  cd real-servers/_repo && git sparse-checkout set src/filesystem`);
  console.log(`(real-servers/ is gitignored — not part of this repo)`);
  throw new Error("real server not cloned");
}

const verdicts = await runPipeline(SERVER_PATH);
console.log(`Found ${verdicts.length} tool(s) in real server.\n`);
for (const v of verdicts) console.log(format(v) + "\n");

## Comparison vs MCPDiFF

| Property | MCPDiFF | This v0 |
|---|---|---|
| Sound? | No — arbitrary 0.61 threshold | Yes (within v0 fragment) — set containment is decidable |
| Explainable? | No — just a similarity score | Yes — every undeclared leaf has source-line witnesses (cell 7) |
| LLM-free? | No — LLM summary per tool | Yes — pure static analysis |
| Speed | Slow (LLM API calls) | Fast (one TS compiler pass) |
| Novel | NLP similarity to MCP | Formal capability model with reuse from Deno + Node permission systems |

Each row of "Yes" above is paid for by a specific cell of this notebook: cell 4 shows soundness on a clean tool (no false positive); cell 7 shows explainability (witnesses); cells 4–6 are LLM-free and complete in milliseconds.

## v0 limitations — read this honestly

The verdicts above are sound *only on the language fragment v0 supports*. Concretely:

1. **Cross-module recursion.** ✅ Resolved in v0.5. The analyser now indexes function declarations across every source file in the program and resolves cross-file `Identifier` callees via the TS type checker. See `demo-servers/subtle-multifile.ts` for a verified cross-file detection.
1. **Bare-name / aliased imports.** ✅ Resolved in v0.6. `import { readFile } from "node:fs/promises"; readFile(p)` and `import { readFile as rf } from ...; rf(p)` and namespace-star imports (`import * as fsp from ...; fsp.readFile(p)`) are now resolved correctly via a per-file import-alias table built from `buildImportAliases(sf)`.
2. **Dynamic dispatch.** Method calls on `any`-typed values, computed property access (`obj[name]()`), and callbacks are not followed. v1 fix: CHA-style class-hierarchy analysis, type-narrowed callsite resolution.
2. **Async tracking through `.then` chains and bare callbacks.** ✅ Partial in v0.7. Promise-chain callbacks (`.then`/`.catch`/`.finally`) and timer schedulers (`setTimeout`, `setInterval`, `setImmediate`, `queueMicrotask`, `process.nextTick`) are now followed. Array iteration callbacks (`.map`, `.forEach`, etc.) remain on the roadmap — the false-positive rate is too high to include without type narrowing.
3. **Description NLP.** `parseDescription.ts` is a regex/keyword map. Real natural language has synonyms, negation, domain words. v1 fix: AutoCog-style NLP pipeline; the interface (`parse(string) → Set<Leaf>`) does not change.
4. **Sink catalog coverage.** v0.6 ships ~55 entries adding Prisma flow-sensitive detection, MongoDB, Redis, ioredis, nodemailer, and node-fetch. The long tail remains community-contribution territory.

Each limitation names a specific interface to swap. The architecture is the contract that lets v1 do that without rewriting unrelated code.

## v1 roadmap (in cost order)

1. Cross-module call resolution via `ts.createProgram` project-graph.
2. ✅ **Partial** in v0.5: ENV leaf split out of READ. Remaining (`sys`, `ffi`, `import`) deferred.
3. ✅ **v0.6**: Expand sink catalog to popular npm libs — added Prisma flow-sensitive detector, MongoDB/Mongoose, Redis/ioredis, nodemailer, node-fetch. Axios entries corrected to source-form keys. Named-import and namespace-import aliasing resolves bare-name calls like `readFile(p)` from `import { readFile } from "node:fs/promises"`.
4. Replace regex parser with AutoCog-style NLP.
5. CHA-grade soundness for class-hierarchy method dispatch.
6. Python adapter behind the same `extractActual` interface — unblocks `mcpx-py` and the proposal's other named real-world examples.
7. ✅ **Partial in v0.7**: Async tracking through `.then`/`.catch`/`.finally` chains and timer schedulers (`setTimeout`, `setInterval`, `setImmediate`, `queueMicrotask`, `process.nextTick`). Array iteration callbacks (`.map`, `.forEach`, etc.) still on roadmap.
8. Replicate MCPDiFF's 10,240-server corpus and cross-check verdicts against their labels.
9. Sub-leaf granularity (Option B from brainstorming) once the corpus tells us which distinctions matter.